#### Visualize Supplemental Table 3 from Zheng et al. 2024 ("Summary of genomic risk loci") and Supplementary Table 8 ("Effector gene prioritization")

In [1]:
import pandas as pd
import numpy as np
from collections import Counter

In [2]:
GWAS_df = pd.read_csv("Zheng_et_al_2024_Supplemental_Table_3_csv_format.csv", header = 0)
print(GWAS_df.shape)

(104, 39)


In [3]:
GWAS_df.head()

,Locus,Chr,Start BP,End BP,rsID,Pos (hg19),EA,OA,EA Freq,Unnamed: 9,...,i2 het.1,Phet.1,Beta.2,A1 freq.2,SE.2,P.2,N case.2,N total.2,i2 het.2,Phet.2
0,1,1,1644107,2644107,rs2503715,2144107,A,G,0.1367,False,...,42.3,3.818000e-02,0.141273,0.1272,0.039028,0.000295,3581,447335,63.9,0.040120
1,2,1,2741254,3741254,rs79548216,3241254,A,C,0.9019,True,...,0.0,8.311000e-01,-0.143129,0.9040,0.041217,0.000516,4084,449045,0.0,0.622200
2,3,1,5778414,6778414,rs709209,6278414,A,G,0.6522,False,...,0.0,5.900000e-01,-0.096355,0.6419,0.023516,0.000042,5619,434759,0.0,0.495700
3,4,1,15839772,16844730,rs1763604,16339772,C,G,0.4074,False,...,NaN,NaN,-0.193142,0.4020,0.022128,0.000000,6001,455291,80.5,0.000103
4,4,1,15839772,16844730,rs945418,16344730,C,T,0.6785,False,...,79.2,2.024000e-10,0.193334,0.6790,0.023308,0.000000,6001,455291,80.3,0.000116


### filter to just those that are genome-wide significant GWAS DCM

In [8]:
GWAS_df = GWAS_df[GWAS_df['Genome-wide significant GWAS DCM'] == True]

In [9]:
effector_genes_df = pd.read_csv("Zheng_Table_S8_effector_gene_scoring.csv")
effector_genes_df.head()

,Gene,Locus,ABC,TWAS,PoPS,Colocalisation,V2G,Coding variant,Nearest,Mendelian disease causing gene,Total score
0,C1orf86,1,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,1
1,FAAP20,1,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,1
2,MORN1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,PEX10,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,PRKCZ,1,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,1


In [10]:
top_effector_gene_per_locus = effector_genes_df.loc[effector_genes_df.groupby("Locus")["Total score"].idxmax()]

In [11]:
top_effector_gene_per_locus.shape

(80, 11)

In [12]:
top_effector_gene_per_locus.head()

,Gene,Locus,ABC,TWAS,PoPS,Colocalisation,V2G,Coding variant,Nearest,Mendelian disease causing gene,Total score
0,C1orf86,1,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,1
10,PRDM16,2,NaN,NaN,True,NaN,True,NaN,True,NaN,3
16,RNF207,3,NaN,NaN,NaN,NaN,True,True,True,NaN,3
21,HSPB7,4,NaN,NaN,True,True,NaN,True,True,NaN,4
26,MAST2,5,NaN,NaN,True,NaN,NaN,True,NaN,NaN,2


#### 104 SNPs in 80 loci, with corresponding score effector genes

In [13]:
#Counter(GWAS_df.Locus)

### Left join to get the GWAS loci and their candidate effector genes

In [14]:
joined_GWAS_df = GWAS_df.merge(top_effector_gene_per_locus, on = "Locus", how = "left")
joined_GWAS_df.head()

,Locus,Chr,Start BP,End BP,rsID,Pos (hg19),EA,OA,EA Freq,Unnamed: 9,...,Gene,ABC,TWAS,PoPS,Colocalisation,V2G,Coding variant,Nearest,Mendelian disease causing gene,Total score
0,4,1,15839772,16844730,rs1763604,16339772,C,G,0.4074,False,...,HSPB7,NaN,NaN,True,True,NaN,True,True,NaN,4
1,6,1,61377378,62377378,rs2103883,61877378,A,G,0.5438,True,...,NFIA,NaN,NaN,NaN,NaN,True,NaN,True,NaN,2
2,9,1,178892863,179892863,rs61822778,179392863,A,G,0.0183,True,...,AXDND1,NaN,NaN,NaN,NaN,True,NaN,True,NaN,2
3,12,1,236352282,237353167,rs12724121,236852282,A,T,0.3775,False,...,ACTN2,NaN,NaN,True,True,True,NaN,True,True,5
4,13,2,11056106,12056106,rs3922995,11556106,A,G,0.3533,True,...,LINC00570,NaN,True,NaN,NaN,NaN,NaN,True,NaN,2


#### Get the location of these SNPs, which are in GRCh37 and the prioritized gene. Save this as a bed file for liftover

In [10]:
GWAS_bed_df = joined_GWAS_df[["Chr", "Pos (hg19)", "rsID", "Gene", "Total score"]]
GWAS_bed_df.head()

,Chr,Pos (hg19),rsID,Gene,Total score
0,1,2144107,rs2503715,C1orf86,1
1,1,3241254,rs79548216,PRDM16,3
2,1,6278414,rs709209,RNF207,3
3,1,16339772,rs1763604,HSPB7,4
4,1,16344730,rs945418,HSPB7,4


In [11]:
GWAS_bed_df = GWAS_bed_df.rename(columns = {"Chr": "chr", 
                                           "Pos (hg19)": "start",
                                           "Total score": "total_score"})

In [12]:
GWAS_bed_df['end'] = GWAS_bed_df['start']
GWAS_bed_df = GWAS_bed_df[["chr", "start", "end", "rsID", "Gene", "total_score"]]
GWAS_bed_df

,chr,start,end,rsID,Gene,total_score
0,1,2144107,2144107,rs2503715,C1orf86,1
1,1,3241254,3241254,rs79548216,PRDM16,3
2,1,6278414,6278414,rs709209,RNF207,3
3,1,16339772,16339772,rs1763604,HSPB7,4
4,1,16344730,16344730,rs945418,HSPB7,4
...,...,...,...,...,...,...
99,20,33634479,33634479,rs13042358,TRPC4AP,2
100,21,30571669,30571669,rs8134232,BACH1,2
101,21,30530131,30530131,rs62222424,BACH1,2
102,21,40029045,40029045,rs796217035,ERG,3


In [14]:
GWAS_bed_df.head(30)

,chr,start,end,rsID,Gene,total_score
0,1,2144107,2144107,rs2503715,C1orf86,1
1,1,3241254,3241254,rs79548216,PRDM16,3
2,1,6278414,6278414,rs709209,RNF207,3
3,1,16339772,16339772,rs1763604,HSPB7,4
4,1,16344730,16344730,rs945418,HSPB7,4
5,1,46021630,46021630,rs2993263,MAST2,2
6,1,61877378,61877378,rs2103883,NFIA,2
7,1,78623626,78623626,rs17391694,NEXN,4
8,1,155738044,155738044,rs12081192,GON4L,4
9,1,179392863,179392863,rs61822778,AXDND1,2


In [13]:
# one of the SNPs lacks an rsid, so fill this column
GWAS_bed_df = GWAS_bed_df.fillna("none")

### Reformat properly for `liftover`

In [14]:
GWAS_bed_df['chr'] = "chr" + GWAS_bed_df['chr'].astype(str)

In [15]:
GWAS_bed_df.head()

,chr,start,end,rsID,Gene,total_score
0,chr1,2144107,2144107,rs2503715,C1orf86,1
1,chr1,3241254,3241254,rs79548216,PRDM16,3
2,chr1,6278414,6278414,rs709209,RNF207,3
3,chr1,16339772,16339772,rs1763604,HSPB7,4
4,chr1,16344730,16344730,rs945418,HSPB7,4


In [16]:
GWAS_bed_df.shape

(104, 6)

### Save to csv, install liftover to liftover from hg19 to hg38

- conda install bioconda::ucsc-liftover

In [17]:
# don't include header, since liftOver doesn't want this
GWAS_bed_df.to_csv("Zheng_GWAS_hits_GRCh37.bed", sep="\t", index=False, header=False)